<h1 align="center">🔎 LLMPreSS · Interactive Semantic Search</h1>

<h3 align="center">Explore LLM pretraining data, semantically—right from your browser.</h3>

<p align="center">Discover relevant passages across the <strong>ClimbMix pretraining corpus</strong>, even when they describe your question or topic using different words.</p>

<p align="center"><strong>No local installation. No dedicated search server. No GPU required.</strong></p>

<p align="center"><a href="https://huggingface.co/collections/JamshidJDMY/llmpress">📚 Explore published indexes</a> · <a href="https://github.com/my-unknown-account/LLMPreSS">💻 LLMPreSS on GitHub</a></p>

---

### ✨ What happens in this notebook?

**Install LLMPreSS → Load a compressed semantic index → Search passages → Explore and export results**

Everything runs in the notebook environment: LLMPreSS downloads the selected FAISS index and BGE query encoder into the **Colab runtime**, performs semantic retrieval there, and uses the remote passage API **only to fetch the matched text and available source metadata**. You do not need to download these large index files onto your own computer.

> **💡 Before you begin:** Run the cells from top to bottom. The example uses **ClimbMix · Compact**. A CPU runtime is sufficient; a GPU is optional for query encoding (**Runtime → Change runtime type**). FAISS vector search runs on the CPU. Choose a runtime with enough disk space and RAM for the index.


## ① Install LLMPreSS

Start by installing the Python package and the interactive search widgets in this notebook environment. **Nothing needs to be installed on your laptop or desktop.**

Run the installation cell below once. If you upgrade a package during the same session and encounter import issues, use **Runtime → Restart session**, then rerun the remaining cells.


In [ ]:
%pip install -q "git+https://github.com/my-unknown-account/LLMPreSS.git" ipywidgets

## ② Load the pretraining index

The next cell prepares everything needed for semantic retrieval:

- **Corpus:** `ClimbMix` — the pretraining collection to explore.
- **Edition:** `compact` — the smaller index edition; you can change it to `quality` if you prefer the higher-fidelity version.
- **Storage and encoding:** Index files are downloaded into the notebook runtime when needed, and the matching BGE encoder is loaded automatically.

**The download is a one-time step per available cache.** After loading, you can run as many queries as you like without reinitializing the index. If a temporary Colab runtime is reset, its downloaded files may need to be fetched again.

> **📦 Storage note:** Even compressed indexes are substantial. Make sure your runtime has sufficient free disk space and memory for the corpus and edition you select.


In [ ]:
from pathlib import Path
import torch
from llmpress import SemanticIndex

corpus = "ClimbMix"
edition = "compact"
index_dir = Path("/content/llmpress-index") if Path("/content").exists() else Path("llmpress-index")
device = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Query encoder device: {device}")
index = SemanticIndex(
    corpus,
    edition=edition,
    index_dir=index_dir,
    verbose=True,
    threads=32,
    device=device,
)
print("Ready. Run the next cell to open the search interface.")

## ③ Search semantically

**Ask a question, describe a concept, or investigate a research topic.** LLMPreSS retrieves passages based on semantic similarity, so your query does not need to use the exact wording found in the corpus.

Use the interactive controls below to enter a query and choose how many matches to retrieve (**1–10,000**, with **1,000** selected by default). The notebook displays the **top 20 passages** for easy reading, including their text, similarity scores, and available source details.

**What happens when you click Search passages?** Your query is encoded and matched against the FAISS index in the current runtime. Only the resulting passage IDs are sent to the remote API to retrieve their text and metadata; **the API does not perform semantic search**.

> **🔬 Research tip:** Try a paraphrased question, an obscure entity, or an alternative description of a familiar concept. Similarity scores help rank passages; they are **not probabilities or proof of training-data memorization**. You can search again without reloading the index.


In [ ]:
#@title Search passages { display-mode: "form" }
import html
import ipywidgets as widgets
from IPython.display import HTML, display

query_box = widgets.Textarea(
    value="How do plants adapt to drought?",
    placeholder="Ask a question or describe a topic…",
    description="Query:",
    layout=widgets.Layout(width="100%", height="90px"),
)
top_k = widgets.IntSlider(value=1000, min=1, max=10000, description="Results:")
search_button = widgets.Button(description="Search passages", button_style="primary", icon="search")
output = widgets.Output()
results = []
last_query = None

def run_search(_):
    global results, last_query
    query = query_box.value.strip()
    output.clear_output(wait=True)
    with output:
        if not query:
            print("Enter a question or topic first.")
            return
        search_button.disabled = True
        results = []
        try:
            print("Searching…")
            results = index.search(query, top_k=top_k.value)
            last_query = query
            output.clear_output(wait=True)
            if not results:
                print("No passages found.")
            if results:
                print(f"Showing top {min(20, len(results))} of {len(results):,} retrieved passages. Export JSON to save all results.")
            for result in results[:20]:
                title = html.escape(str(result.passage_id or result.numeric_id))
                text = html.escape(result.text or "Passage text unavailable.")
                source = " · ".join(
                    f"{key}: {value}" for key, value in (
                        ("Corpus", result.corpus),
                        ("Source", result.source_rel_path),
                        ("Group", result.source_group),
                        ("Record", result.source_record_index),
                    ) if value is not None
                )
                display(HTML(
                    f'<div style="border:1px solid #94a3b8;border-radius:12px;padding:18px;margin:12px 0">'
                    f'<strong>{result.rank}. {title}</strong> · Score: {result.score:.4f}'
                    f'<p style="white-space:pre-wrap;line-height:1.6">{text}</p>'
                    f'<small>{html.escape(source)}</small></div>'
                ))
        except Exception as error:
            print(f"Search failed: {error}")
        finally:
            search_button.disabled = False

search_button.on_click(run_search)
display(widgets.VBox([query_box, top_k, search_button, output]))

## ④ Download your results (optional)

Want to analyze the retrieved passages later? After running a successful search, execute the cell below to export **your latest query and every retrieved result** as a structured JSON file—not just the 20 passages shown in the preview.

In **Google Colab**, the file downloads directly through your browser. In another notebook environment, it is saved as `llmpress-results.json` in the current working directory. The export includes available passage identifiers, scores, text, and source metadata.


In [ ]:
import json
from pathlib import Path

if not results:
    raise ValueError("Run a successful search before exporting.")
export_path = Path("llmpress-results.json")
payload = {
    "query": last_query,
    "results": [result.to_dict() for result in results],
}
export_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
try:
    from google.colab import files
except ImportError:
    print(f"Saved to {export_path.resolve()}")
else:
    files.download(str(export_path))